Deutsch Algorithm using proper balanced and constant function
During a run make the fn randomly constant or balanced and verify the output
Use different set of gates to implement the Uf for all the 4 types of constant and balanced functions which are
1. f(0)=f(1)=0 this is F1(x)
2. f(0)=f(1)=1 this is F2(x)
3. f(0)=0, f(1)=1 this is F3(x)
4. f(0)=1, f(0)=0 this is F4(x)


In [30]:
import numpy as np
import Utilities as utils
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import StatevectorSimulator, QasmSimulator, Aer

S_simulator = StatevectorSimulator()
M_simulator = QasmSimulator()


In [31]:
# Defining the classical and quantum black box functions

def classical_blackbox():
    '''
    This function returns one of the four functions
    1. f(0)=f(1)=0 this is F1(x)
    2. f(0)=f(1)=1 this is F2(x)
    3. f(0)=0, f(1)=1 this is F3(x)
    4. f(0)=1, f(1)=0 this is F4(x)
    '''

    def F1(x):
        return 0
    def F2(x):
        return 1
    def F3(x):
        return x%2
    def F4(x):
        return (x+1)%2

    functions = [F1, F2, F3, F4]
    return np.random.choice(functions).__name__


def quantum_blackbox(qc, qreg):
    '''
    This function takes a quantum circuit and a quantum register 
    as input and applies one of the four functions

    1. f(0)=f(1)=0 this is F1(x) - No operation. Constant function

    2. f(0)=f(1)=1 this is F2(x) Constant fucntion so 
        Uf|00> = |0XORf(0),0> = |10>
        Uf|01> = Uf|0XORf(1),1> = |11>, 
        Uf|10> = |1XORf(0),0> = |00>, 
        Uf|11>= |1XORf(1),1> = |01>
        This means jusy apply qc.x to first qubit 
        which is the bottom qubit in our representation

    3. f(0)=0, f(1)=1 this is F3(x) Balanced function
        Uf|00> = |0XORf(0),0> = |00>
        Uf|01> = Uf|0XORf(1),1> = |11>, 
        Uf|10> = |1XORf(0),0> = |10>, 
        Uf|11>= |1XORf(1),1> = |01>
        This means apply qc.cx with control as the 
        top qubit and target as the bottom qubit
        Note qc treats top qubit as first qubit 
        opposite to our representation

    4. f(0)=1, f(1)=0 this is F4(x) Balanced function
        Uf|00> = |0XORf(0),0> = |10>
        Uf|01> = Uf|0XORf(1),1> = |01>, 
        Uf|10> = |1XORf(0),0> = |00>, 
        Uf|11>= |1XORf(1),1> = |11>
        this is actually CNOT bar ie control works on 0 from top qubit
        can implement cx first and then apply x to the 
        target qubit which is bottom qubit in our representation
        but index[1] in qreg as per qiskit
    '''

    def F1_Constant(qc, qreg):
        pass
    def F2_Constant(qc, qreg):
        qc.x(qreg[1])
    def F3_Balanced(qc, qreg):
        qc.cx(qreg[0], qreg[1])
    def F4_Balanced(qc, qreg):
        qc.cx(qreg[0], qreg[1])
        qc.x(qreg[1])

    functions = [F1_Constant, F2_Constant, F3_Balanced, F4_Balanced]
    func = np.random.choice(functions)
    func(qc, qreg)
    return func.__name__


print("Classical Black Box Function:", classical_blackbox())
qc = QuantumCircuit()
qreg = QuantumRegister(2)
qc.add_register(qreg)
print("Quantum Black Box Function:", quantum_blackbox(qc, qreg))
print("qc.num_qubits:", qc.num_qubits)
utils.Wavefunction(qc) 


Classical Black Box Function: F1
Quantum Black Box Function: F4_Balanced
qc.num_qubits: 2
1.0 |10> 


In [32]:
# Deutsch's Algorithm

qc = QuantumCircuit()
qreg = QuantumRegister(2)
creg = ClassicalRegister(1)
qc.add_register(qreg)
qc.add_register(creg)
qc.h(qreg[0])
qc.x(qreg[1])
qc.h(qreg[1])
print("State before Quantum Blackbox:")
utils.Wavefunction(qc)

# apply black box function
print(quantum_blackbox(qc, qreg))

print("State after Quantum Blackbox:")
utils.Wavefunction(qc)

qc.h(qreg[0])
qc.measure(qreg[0], creg[0])

counts = M_simulator.run(qc, shots=1).result().get_counts()

result = int(next(iter(counts)))

if(result == 0):
    print("The function is constant")
else:
    print("The function is balanced")

State before Quantum Blackbox:
0.5 |00> 0.5 |01> -0.5 |10> -0.5 |11> 
F3_Balanced
State after Quantum Blackbox:
0.5 |00> -0.5 |01> -0.5 |10> 0.5 |11> 
The function is balanced
